In [ ]:
# LEACE Visualization Notebook
#
# This notebook visualizes the effect of LEACE projection:
# 1. Embed sample texts before and after LEACE
# 2. Visualize embeddings with PCA
# 3. Train linear probes to measure demographic leakage
# 4. Calculate Amnesic Drop metric

import sys
sys.path.append('..')

In [ ]:
# Import dependencies
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.decomposition import PCA

from neuro_stylometry.pollution_guard.embedder import FrozenEmbedder
from neuro_stylometry.pollution_guard.leace import LEACEComputer
from neuro_stylometry.pollution_guard.probe import LinearProbe, compute_amnesic_drop
from neuro_stylometry.data_engine.dataset import SOBRDataset

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Data and Compute Embeddings

Load sample posts and extract embeddings before projection.

In [ ]:
# Load dataset
dataset_path = Path("../artifacts/data/sobr_laptop.arrow")
dataset = SOBRDataset(arrow_path=dataset_path, seed=42)

# Take a sample with known nationalities
sample_size = 200
table = dataset.table.slice(0, sample_size)

# Filter for non-null nationality labels
nationality_col = table["nationality"].to_pandas()
valid_mask = nationality_col.notna()
valid_indices = np.where(valid_mask)[0]

if len(valid_indices) < 50:
    print(f"Warning: Only {len(valid_indices)} samples with nationality labels")
    # Take more samples
    table = dataset.table.slice(0, 1000)
    nationality_col = table["nationality"].to_pandas()
    valid_mask = nationality_col.notna()
    valid_indices = np.where(valid_mask)[0][:200]

# Extract valid samples
posts = [table["post"][i].as_py() for i in valid_indices]
labels_series = nationality_col.iloc[valid_indices]
labels_int, label_names = pd.factorize(labels_series)
labels = torch.tensor(labels_int, dtype=torch.long)

print(f"Loaded {len(posts)} posts with {len(label_names)} unique nationalities")
print(f"Label distribution:\n{pd.Series(labels_int).value_counts().head()}")

# Initialize embedder
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\nUsing device: {device}")

embedder = FrozenEmbedder(
    model_name="roberta-base",
    device=device,
    max_length=512,
)

# Embed posts
print("\nEmbedding posts...")
embeddings_before = embedder.embed_texts(posts, batch_size=4, show_progress=True)

print(f"Embeddings shape: {embeddings_before.shape}")

## 2. Compute LEACE Projection

Compute projection matrix to remove nationality information.

In [ ]:
# Initialize LEACE computer
leace = LEACEComputer(
    embedding_dim=embedder.get_embedding_dim(),
    regularization=1e-5,
    device=device,
)

# Compute projection matrix
print("Computing LEACE projection matrix...")
projection_matrix = leace.compute_projection(embeddings_before, labels)

# Apply projection
embeddings_after = embeddings_before @ projection_matrix.T

print(f"Projection matrix shape: {projection_matrix.shape}")
print(f"Embeddings after projection: {embeddings_after.shape}")

## 3. Visualize with PCA

Use PCA to visualize embeddings before and after LEACE projection.

In [ ]:
# Apply PCA for visualization
pca = PCA(n_components=2)

# Transform before
embeddings_before_np = embeddings_before.cpu().numpy()
pca_before = pca.fit_transform(embeddings_before_np)

# Transform after
embeddings_after_np = embeddings_after.cpu().numpy()
pca_after = pca.fit_transform(embeddings_after_np)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot before
for label_idx, label_name in enumerate(label_names[:5]):  # Top 5 nationalities
    mask = labels_int == label_idx
    axes[0].scatter(
        pca_before[mask, 0],
        pca_before[mask, 1],
        label=label_name,
        alpha=0.6,
        s=50,
    )

axes[0].set_xlabel('PC1', fontsize=12)
axes[0].set_ylabel('PC2', fontsize=12)
axes[0].set_title('Embeddings Before LEACE', fontsize=14, fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot after
for label_idx, label_name in enumerate(label_names[:5]):
    mask = labels_int == label_idx
    axes[1].scatter(
        pca_after[mask, 0],
        pca_after[mask, 1],
        label=label_name,
        alpha=0.6,
        s=50,
    )

axes[1].set_xlabel('PC1', fontsize=12)
axes[1].set_ylabel('PC2', fontsize=12)
axes[1].set_title('Embeddings After LEACE', fontsize=14, fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Note: Clusters should be less separated after LEACE projection")

## 4. Compute Amnesic Drop

Train linear probes to measure demographic leakage before and after LEACE.

In [ ]:
# Compute amnesic drop
print("Computing Amnesic Drop metric...\n")

acc_before, acc_after, amnesic_drop = compute_amnesic_drop(
    embeddings_before=embeddings_before,
    embeddings_after=embeddings_after,
    labels=labels,
    train_split=0.8,
    random_state=42,
)

# Visualize results
fig, ax = plt.subplots(figsize=(10, 6))

metrics = ['Accuracy Before', 'Accuracy After']
values = [acc_before * 100, acc_after * 100]
colors = ['#d32f2f', '#388e3c']

bars = ax.bar(metrics, values, color=colors, alpha=0.7, edgecolor='black')

# Add value labels
for bar, value in zip(bars, values):
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width() / 2.,
        height + 1,
        f'{value:.1f}%',
        ha='center',
        va='bottom',
        fontsize=14,
        fontweight='bold',
    )

ax.set_ylabel('Accuracy (%)', fontsize=12)
ax.set_title('Linear Probe Accuracy: Before vs After LEACE', fontsize=14, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary
print("\n" + "=" * 60)
print("AMNESIC DROP SUMMARY")
print("=" * 60)
print(f"Accuracy Before LEACE: {acc_before:.1%}")
print(f"Accuracy After LEACE:  {acc_after:.1%}")
print(f"Amnesic Drop:          {amnesic_drop:.1%}")
print()

if amnesic_drop > 0.30:
    print("✓ Target achieved: Amnesic Drop > 30%")
else:
    print(f"✗ Target not met: Amnesic Drop {amnesic_drop:.1%} < 30%")

print("=" * 60)